# Native QAOA: MaxCut and two Knapsack models

Complete source-sized QCE part II workflow. Three layers and six radian angles. Local exact-mean native COBYLA replaces the provider optimizer. Actual query caps are40/50, and the final actual query is selected. No optimizer trajectory or performance equivalence is claimed. A budget stop is not convergence. Quantum phases and classical objective penalties deliberately differ.


## MaxCut warm-up

The graph has edges(0,1) and(0,2). Array element0 is the low bit of the decoded word. RX(beta) is retained without a factor of two. Each classical binding produces a separately synthesized program.


In [ ]:
from io import BytesIO
from typing import Annotated

from IPython.display import Image, display
from matplotlib import pyplot as plt

import flagquantum as fq
from examples.synthesis.tutorials.workshops.algo_design_qce_tutorial.algo_design_qce_tutorial_part_ii.overview.model import (
    BINDINGS,
    INITIAL,
    feasible_value,
)
from examples.synthesis.tutorials.workshops.algo_design_qce_tutorial.algo_design_qce_tutorial_part_ii.overview.presentation import (
    plot_convergence,
    sample_table,
)
from examples.synthesis.tutorials.workshops.algo_design_qce_tutorial.algo_design_qce_tutorial_part_ii.overview.verification import (
    verify_program,
)
from examples.synthesis.tutorials.workshops.algo_design_qce_tutorial.algo_design_qce_tutorial_part_ii.overview.workflow import (
    Optimization,
    optimize,
    statistics,
)
from flagquantum import synth
from flagquantum.synth import (
    RX,
    Output,
    QArray,
    QBit,
    QNum,
    allocate,
    control,
    hadamard_transform,
    phase,
    qfunc,
)


@qfunc
def maxcut(params: list[float], v: Output[QArray[QBit]]) -> None:
    allocate(3, v)
    hadamard_transform(v)
    for layer in range(3):
        phase(
            -(v[0] + v[1] - 2 * v[0] * v[1] + v[0] + v[2] - 2 * v[0] * v[2]),
            params[layer],
        )
        for bit in range(3):
            RX(params[3 + layer], v[bit])


maxcut_program = synth.synthesize(
    maxcut,
    classical_inputs={"params": list(BINDINGS[0])},
    memory_limit_bytes=64 * 1024**2,
)
print(verify_program("maxcut", BINDINGS[0], maxcut_program))
print(maxcut_program.report["resources"])

print(
    "MaxCut normalization:",
    sum(
        maxcut_program.probabilities(fq.run(maxcut_program), registers=("v",)).values()
    ),
)


## Initial schedule and actual MaxCut observations

Gamma starts at[0,pi/2,pi], beta at[pi,pi/2,0]. The provider sample count was implicit; this native workflow explicitly uses1000 draws. Reference optima label the problem only; no counts or optimum are injected.


In [ ]:
print("Initial schedule:", INITIAL)
maxcut_run = optimize("maxcut")
print("Actual objective calls:", len(maxcut_run.evaluations))
print(maxcut_run.termination, "Converged:", maxcut_run.converged)
print("Last query:", maxcut_run.selected)
print(sample_table(maxcut_run))


## Exercise1 reference solution: problem variables

Native named a/b ports replace the source quantum struct fields; widths remain3/2. These classical functions operate on decoded values. The source exercise placeholders and final solution are represented by this labeled executable solution.


In [ ]:
print("Feasible assignment(3,2):", feasible_value(3, 2))
print("Infeasible assignment(3,3):", feasible_value(3, 3))
assert feasible_value(3, 2) == 19
assert feasible_value(3, 3) == 0


## Shared optimization and readout

Quantum slack phase is -value+2*(weight+slack-12)**2. The optimizer for both Knapsack models instead minimizes -value+4*max(weight-12,0). Its score is not the phase expectation. Statistics use1000 actual draws; each final small table uses10 separate actual draws. The native COBYLA callback enforces the source40/50 limits because native trust-region iterations and objective evaluations are different units.


In [ ]:
def show_run(run: Optimization) -> None:
    print("Actual objective calls:", len(run.evaluations))
    print(run.termination, "Converged:", run.converged)
    print("Last query:", run.selected)
    print(statistics(run.statistics_samples))
    print(sample_table(run))
    figure = plot_convergence(run)
    buffer = BytesIO()
    figure.savefig(buffer, format="png", dpi=130)
    display(Image(buffer.getvalue()))
    plt.close(figure)


## Exercise2 reference solution: slack penalty

Retain3/2/4-bit a/b/slack registers, Hadamards on all nine bits, and three cost/mixer layers. The histogram/table retains slack but feasibility statistics ignore it.


In [ ]:
@qfunc
def penalty(
    params: list[float],
    a: Output[Annotated[QNum, 3]],
    b: Output[Annotated[QNum, 2]],
    slack: Output[Annotated[QNum, 4]],
) -> None:
    allocate(a)
    allocate(b)
    allocate(slack)
    hadamard_transform(a)
    hadamard_transform(b)
    hadamard_transform(slack)
    for layer in range(3):
        phase(-(3 * a + 5 * b) + 2 * (2 * a + 3 * b + slack - 12) ** 2, params[layer])
        for bit in range(3):
            RX(params[3 + layer], a[bit])
        for bit in range(2):
            RX(params[3 + layer], b[bit])
        for bit in range(4):
            RX(params[3 + layer], slack[bit])


penalty_program = synth.synthesize(
    penalty,
    classical_inputs={"params": list(BINDINGS[0])},
    memory_limit_bytes=64 * 1024**2,
)
print(verify_program("penalty", BINDINGS[0], penalty_program))
print(penalty_program.report["resources"])
penalty_run = optimize("penalty", seed=1972)
show_run(penalty_run)


## Exercise3 reference solution: exact feasible phase

The condition weight<=12 controls -value in phase; there is no slack register. Keep both five-bit logical data and full global phase in the independent Circuit check. This sample is not guaranteed to converge faster or score higher.


In [ ]:
@qfunc
def feasible_cost(gamma: float, a: QNum, b: QNum) -> None:
    control(2 * a + 3 * b <= 12, lambda: phase(-(3 * a + 5 * b), gamma))


@qfunc
def feasible(
    params: list[float], a: Output[Annotated[QNum, 3]], b: Output[Annotated[QNum, 2]]
) -> None:
    allocate(a)
    allocate(b)
    hadamard_transform(a)
    hadamard_transform(b)
    for layer in range(3):
        feasible_cost(params[layer], a, b)
        for bit in range(3):
            RX(params[3 + layer], a[bit])
        for bit in range(2):
            RX(params[3 + layer], b[bit])


feasible_program = synth.synthesize(
    feasible,
    classical_inputs={"params": list(BINDINGS[0])},
    memory_limit_bytes=64 * 1024**2,
)
print(verify_program("feasible", BINDINGS[0], feasible_program))
print(feasible_program.report["resources"])
feasible_run = optimize("feasible", seed=1974)
show_run(feasible_run)


## Solution1 replay

Repeat the source classical feasibility check. All a=0..7,b=0..3 assignments are enumerated only to label the classical optimum; they do not supply quantum samples.


In [ ]:
assert feasible_value(3, 2) == 19
assert feasible_value(3, 3) == 0
optimum = max(feasible_value(a, b) for a in range(8) for b in range(4))
print("Classical reference optimum:", optimum)


## Solution2 replay

Repeat the full bounded slack optimization and actual10-draw final table, as the source solution cell does. Random readout seeds are explicit local policy.


In [ ]:
penalty_solution = optimize("penalty", seed=1976)
show_run(penalty_solution)


## Solution3 replay

Repeat the full bounded feasible-phase optimization and actual10-draw final table. This records the last observed query, not an injected optimum. Resources above count synthesized instructions before Runtime compilation; no hardware, speedup or optimality claim is made.


In [ ]:
feasible_solution = optimize("feasible", seed=1978)
show_run(feasible_solution)
